# Selección y Model Registry

| | |
|---|---|
| **Entrega** | PSet 3, sección 8 de `CD_PSet_3.pdf` |
| **Código** | `src/seleccion.py` |
| **Hold-out** | se abre **una sola vez**: el resultado queda en `docs/holdout_finalistas.csv` y en corridas `final` de MLflow; si ya existe, el notebook lo lee y no vuelve a evaluar |

**Sección 8 del PDF.** Elige el ganador con evidencia en los experimentos:
1. Compara el ganador con el segundo mejor usando el promedio y la desviación de la CV. ¿La diferencia supera la
   variación entre pliegues?
2. Evalúa ambos una sola vez en el conjunto de prueba.
3. Calcula la métrica de negocio.
4. Compara el costo y la explicabilidad: tiempo de entrenamiento e inferencia. ¿Puedes explicar por qué el modelo hace
   una predicción?

Registra el ganador en el Model Registry de MLflow (`<proyecto>-modelo`, alias `champion` y `challenger`), recárgalo en
un notebook limpio (`06_champion.ipynb`) y exporta las corridas a `docs/corridas.csv`.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.datos import cargar
from src.pipeline import preparar, dividir, FEATURES, TARGET, pliegues
from src.negocio import RiesgoDiario, portafolio_universo, serie_indice, resumen
from src import seleccion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True})
obt, metadata = cargar()
datos = preparar(obt)
train, holdout = dividir(datos)
dia, metadata_dia = cargar(grano='dia')
riesgo = RiesgoDiario(dia)
print({'fuente': metadata['tabla'], 'sha256': metadata['sha256'], 'filas_obt': len(obt), 'entrenamiento': len(train), 'holdout_reservado': len(holdout)})

{'fuente': 'PORTAFOLIO_P3.OBT.OBT_ACCION_MES', 'sha256': 'c356891723976bceaa7447fe9682902ea520e46ad5bc9490ba43093a228f228f', 'filas_obt': 87178, 'entrenamiento': 46851, 'holdout_reservado': 11326}


## 8.1 Finalistas por la métrica técnica (MAE de CV)

Se juntan los candidatos de la sección 6 y la mejor configuración de cada tuning (sección 7), y se ordenan por el MAE
de CV. Las líneas base no compiten: son la vara.

In [2]:
f = seleccion.finalistas()
display(f.drop(columns=['model_uri'], errors='ignore').round(4))
registrables = f[f.registrable]
CHAMPION, CHALLENGER = registrables.iloc[0], registrables.iloc[1]
fuera = f.loc[:registrables.index[1]][~f.loc[:registrables.index[1]].registrable]
if len(fuera):
    print('Aviso: entre los dos primeros hay modelos sin Pipeline registrable (fundacionales en zero-shot):',
          fuera.modelo.tolist(), '-> se reportan, pero champion y challenger deben poder cargarse desde el Model Registry.')
print(f'champion: {CHAMPION.modelo} ({CHAMPION.version}), MAE CV {CHAMPION.mae_cv_media:.4f} ± {CHAMPION.mae_cv_std:.4f}')
print(f'challenger: {CHALLENGER.modelo} ({CHALLENGER.version}), MAE CV {CHALLENGER.mae_cv_media:.4f} ± {CHALLENGER.mae_cv_std:.4f}')

,modelo,version,familia,mae_train_cv,mae_cv_media,mae_cv_std,brecha_mae,ic_media,negocio_sharpe_neto,negocio_drawdown_max,segundos_entrenamiento,run_id,registrable
0,M3_lightgbm,ajustado (Optuna),boosting,0.0478,0.0636,0.0163,0.0157,0.0040,0.3019,-0.3540,8.8955,f7596a565f8a458492057238f245cf89,True
1,M4_ridge,ajustado (Optuna),lineal,0.0524,0.0636,0.0161,0.0112,-0.0063,0.4213,-0.2452,5.8062,75f75a7030154008a54e502bf2e4d18b,True
2,M3b_xgboost,ajustado (Optuna),boosting,0.0513,0.0640,0.0169,0.0127,-0.0089,0.2664,-0.2685,1.4338,b3c5c84d6ef44b10a16ee01a7fd3d298,True
3,M8_chronos,por defecto,fundacional,0.0550,0.0670,0.0175,0.0120,-0.0253,0.5682,-0.2212,0.0000,cae84206dd404618b1812618eb5b6e3b,False
4,M9_timesfm,por defecto,fundacional,0.0578,0.0687,0.0171,0.0109,-0.0110,0.4569,-0.2215,0.0000,eff5db0428384e13979a22a98a8e0e78,False
5,M10_moirai,por defecto,fundacional,0.0563,0.0693,0.0192,0.0130,-0.0171,0.4302,-0.2278,0.0000,d3cdca347b8c49dbafe3636a5a55bfe1,False
6,M2_random_forest,por defecto,bagging,0.0149,0.0757,0.0247,0.0608,-0.0001,0.2014,-0.3392,205.9746,3e899513e63e42d48ea01927c443240d,True
7,M5_mlp,por defecto,red_neuronal,0.0449,0.0896,0.0189,0.0447,-0.0177,0.2846,-0.3233,16.1736,5056d83e572f42c4a420b6090f5afaed,True


champion: M3_lightgbm (ajustado (Optuna)), MAE CV 0.0636 ± 0.0163
challenger: M4_ridge (ajustado (Optuna)), MAE CV 0.0636 ± 0.0161


## 8.2 ¿La diferencia supera la variación entre pliegues?

In [3]:
comp = seleccion.comparar_pliegues(CHAMPION.run_id, CHALLENGER.run_id)
display(pd.Series(comp).to_frame('champion vs challenger'))
if comp.get('pliegues'):
    print('La diferencia media de MAE (%.4f) %s la desviación entre pliegues del champion (%.4f); gana en %d de %d años.'
          % (comp['diferencia_media'], 'SUPERA' if comp['supera_variacion'] else 'NO supera',
             comp['desv_entre_pliegues_a'], comp['pliegues_que_gana_a'], comp['pliegues']))

/Users/paulehmig/Documents/Personal/USFQ/MaestriaDataScience/src/MSDS_6003/data-science-end-to-end-work/psets/pset-03/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,champion vs challenger
pliegues,7
diferencia_media,0.000056
desv_diferencias,0.002058
pliegues_que_gana_a,2
desv_entre_pliegues_a,0.01628
supera_variacion,False
t_pareado,0.071627


La diferencia media de MAE (0.0001) NO supera la desviación entre pliegues del champion (0.0163); gana en 2 de 7 años.


## 8.3 Hold-out: una sola evaluación

In [4]:
RUTA_HO = ROOT / 'docs/holdout_finalistas.csv'
if RUTA_HO.exists():
    ho = pd.read_csv(RUTA_HO)
    print('El hold-out ya se evaluó (', ho.run_id.tolist(), '): se leen los resultados, no se vuelve a evaluar.')
else:
    filas = [seleccion.evaluar_holdout(CHAMPION.modelo, CHAMPION.model_uri, holdout, metadata, 'champion', riesgo, metadata_dia, CHAMPION.run_id),
             seleccion.evaluar_holdout(CHALLENGER.modelo, CHALLENGER.model_uri, holdout, metadata, 'challenger', riesgo, metadata_dia, CHALLENGER.run_id)]
    ho = pd.DataFrame(filas); ho.to_csv(RUTA_HO, index=False)
cols = ['modelo', 'rol', 'mae_holdout', 'rmse_holdout', 'r2_vs_rw_holdout', 'acierto_direccional_holdout', 'ic_media_holdout',
        'negocio_sharpe_neto_holdout', 'negocio_drawdown_max_holdout', 'negocio_retorno_anual_neto_holdout', 'ms_inferencia_por_fila', 'run_id']
display(ho[[x for x in cols if x in ho]].round(4))

🏃 View run final_M3_lightgbm_champion at: http://localhost:5001/#/experiments/2/runs/50f2d7b9f44048d79506ce8d65ac4c87
🧪 View experiment at: http://localhost:5001/#/experiments/2


🏃 View run final_M4_ridge_challenger at: http://localhost:5001/#/experiments/2/runs/a07387177c294fe8acd7e0fc1bf911b0
🧪 View experiment at: http://localhost:5001/#/experiments/2


,modelo,rol,mae_holdout,rmse_holdout,r2_vs_rw_holdout,acierto_direccional_holdout,ic_media_holdout,negocio_sharpe_neto_holdout,negocio_drawdown_max_holdout,negocio_retorno_anual_neto_holdout,ms_inferencia_por_fila,run_id
0,M3_lightgbm,champion,0.0673,0.0890,-0.1000,0.4996,-0.0009,0.8693,-0.1152,0.1585,0.0045,50f2d7b9f44048d79506ce8d65ac4c87
1,M4_ridge,challenger,0.0635,0.0845,0.0074,0.5454,-0.0092,-0.0188,-0.1720,0.0313,0.0033,a07387177c294fe8acd7e0fc1bf911b0


In [5]:
# Referencias en los mismos meses del hold-out (no son modelos: B1 con la media del entrenamiento, B5 y el S&P 500).
from src.pipeline import metricas_regresion
b1 = metricas_regresion(holdout[TARGET], np.full(len(holdout), train[TARGET].mean()))
refs = {'B1 media (MAE)': b1['mae'],
        'B5 universo S&P 500 ∩ NYSE (Sharpe neto)': resumen(portafolio_universo(holdout))['sharpe_neto'],
        'S&P 500 índice de precio (Sharpe)': resumen(serie_indice(holdout, 'ret_log_sp500_mes_siguiente'))['sharpe_neto']}
display(pd.Series(refs).round(4).to_frame('hold-out 2024-01..2026-10'))

,hold-out 2024-01..2026-10
B1 media (MAE),0.0634
B5 universo S&P 500 ∩ NYSE (Sharpe neto),0.6781
S&P 500 índice de precio (Sharpe),1.0990


## 8.4 Costo y explicabilidad

In [6]:
import mlflow.sklearn
filas = []
for fila, rol in ((CHAMPION, 'champion'), (CHALLENGER, 'challenger')):
    m = mlflow.sklearn.load_model(fila.model_uri)
    est = m.steps[-1][1]
    nombres = m[:-1].get_feature_names_out() if hasattr(m[:-1], 'get_feature_names_out') else None
    if hasattr(est, 'coef_'):
        peso, tipo = np.ravel(est.coef_), 'coeficiente (variables escaladas)'
    elif hasattr(est, 'feature_importances_'):
        peso, tipo = np.asarray(est.feature_importances_, float), 'importancia (ganancia)'
    else:
        peso, tipo = None, 'sin explicación directa'
    if peso is not None and nombres is not None:
        top = pd.Series(peso, index=nombres).sort_values(key=np.abs, ascending=False).head(8).round(4)
        print(f'{rol} {fila.modelo} - {tipo}:'); display(top.to_frame(tipo))
    filas.append({'rol': rol, 'modelo': fila.modelo, 'segundos_entrenamiento_cv': fila.segundos_entrenamiento,
                  'ms_inferencia_por_fila_holdout': float(ho.set_index('rol').loc[rol, 'ms_inferencia_por_fila']),
                  'explicacion': tipo})
display(pd.DataFrame(filas).round(4))

champion M3_lightgbm - importancia (ganancia):


,importancia (ganancia)
dispersion_mes,1428.0
vol_21d,298.0
rango_log_hl_prom_63d,285.0
rango_log_hl_prom_21d,270.0
vol_63d,254.0
vol_garman_klass_21d,222.0
vol_parkinson_21d,216.0
rango_log_hl_mes,183.0


challenger M4_ridge - coeficiente (variables escaladas):


,coeficiente (variables escaladas)
dispersion_mes,0.0057
mes_calendario_obj_11,0.0021
mes_calendario_obj_9,-0.0020
ret_log_nyse_composite_mes,-0.0014
rf_mes,-0.0013
dias_negociados_mes,0.0012
ret_log_sp500_mes,-0.0011
mes_calendario_obj_8,-0.0011


,rol,modelo,segundos_entrenamiento_cv,ms_inferencia_por_fila_holdout,explicacion
0,champion,M3_lightgbm,8.8955,0.0045,importancia (ganancia)
1,challenger,M4_ridge,5.8062,0.0033,coeficiente (variables escaladas)


## 8.5 Model Registry y exportación de corridas

In [7]:
from mlflow import MlflowClient
cliente = MlflowClient()
try:
    alias = {a: cliente.get_model_version_by_alias(seleccion.NOMBRE_MODELO, a).version for a in ('champion', 'challenger')}
    print('Ya registrado:', alias)
except Exception:
    print(seleccion.registrar_en_registry(CHAMPION.model_uri, CHALLENGER.model_uri))
print('Corridas exportadas a', seleccion.exportar_corridas())

Successfully registered model 'portafolio-inversiones-modelo'.
2026/11/18 13:50:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: portafolio-inversiones-modelo, version 1


Created version '1' of model 'portafolio-inversiones-modelo'.
Registered model 'portafolio-inversiones-modelo' already exists. Creating a new version of this model...
2026/11/18 13:50:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: portafolio-inversiones-modelo, version 2


Created version '2' of model 'portafolio-inversiones-modelo'.


{'nombre': 'portafolio-inversiones-modelo', 'champion': '1', 'challenger': '2'}


Corridas exportadas a /Users/paulehmig/Documents/Personal/USFQ/MaestriaDataScience/src/MSDS_6003/data-science-end-to-end-work/psets/pset-03/docs/corridas.csv


**Decisión.**

- **Champion: M3 LightGBM ajustado. Challenger: M4 Ridge ajustado.** Se eligieron por el MAE de CV, la regla
  fijada antes de abrir el hold-out.
- **La diferencia entre ellos no es significativa.** Es de 0,00006 en MAE, contra una desviación entre pliegues de
  0,016. LightGBM gana en 2 de 7 años y el t pareado es 0,07. Ganó por un margen mínimo.
- **Hold-out (2024-01 a 2026-10, evaluado una sola vez):**

| | MAE | Acierto direccional | IC | Sharpe neto | Drawdown máximo |
|---|---|---|---|---|---|
| Champion LightGBM | 0,0673 | 50 % | −0,001 | 0,87 | −11,5 % |
| Challenger Ridge | 0,0635 | 55 % | −0,009 | −0,02 | −17,2 % |
| B1 media | 0,0634 | | | | |
| B5 universo S&P 500 ∩ NYSE | | | | 0,68 | |
| S&P 500 (precio) | | | | 1,10 | |

  - En la métrica técnica **ninguno le gana a B1**: Ridge empata y LightGBM es peor.
  - En el negocio, el portafolio de LightGBM supera a B5, con menor drawdown, pero no al S&P 500. Como su IC es
    prácticamente cero y en CV su Sharpe fue solo 0,30, esa ventaja se reporta **como resultado a confirmar, no como
    evidencia de capacidad predictiva**.
  - No se cambia el champion después de ver el hold-out, para no usarlo como un segundo conjunto de validación.
- **Costo:** los dos son baratos. El entrenamiento en CV toma 8,9 s (LightGBM) y 5,8 s (Ridge); la inferencia,
  menos de 0,005 ms por fila.
- **Explicabilidad:** Ridge se lee directo de sus coeficientes. LightGBM se explica con la importancia por ganancia.
  En ambos domina `dispersion_mes`, la dispersión transversal de los retornos del mes t, que es igual para todas
  las acciones del mes: el modelo ajusta sobre todo el nivel del retorno del mes siguiente, no qué acción rinde más
  que otra. Esto es coherente con un IC cercano a cero.
- **Conclusión:** el champion queda registrado (`portafolio-inversiones-modelo@champion`, versión 1) y se recarga en
  `06_champion.ipynb`, que reproduce exactamente su MAE. Aun así, **no se recomienda usarlo en producción como
  pronosticador**: no supera a B1. El PSet 4 debe probar los modelos M1, M6 y M7 y otra variable objetivo, por
  ejemplo el ranking relativo.
